# NB04 · Semana 8 — Transformación con dbt: silver y gold (Entrega 2)
**Ingeniería de Datos — LCD · UCA Rosario · 2026** · grupo06 · Clima de Sídney (Open-Meteo)

El proyecto dbt está en `dbt/`. **Silver lee de la tabla Delta de bronze** (source `bronze.clima_diario`
con el plugin `delta` de dbt-duckdb), no de un CSV.

In [ ]:
import os, sys
from pathlib import Path
# Raíz del proyecto = carpeta padre de notebooks/ (en el servidor: ~/work/dags)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
import entorno
entorno.info()
import subprocess
def dbt(*args, env=None):
    r = subprocess.run(["dbt", *args, "--profiles-dir", ".", "--no-use-colors"], cwd=RAIZ / "dbt",
                       capture_output=True, text=True, env={**os.environ, **(env or {})})
    print(r.stdout[-6000:], r.stderr[-2000:])
    return r.returncode

## 1. Modelos y linaje
```
source bronze.clima_diario (Delta) ─► stg_clima_diario ─┬─► fct_clima_mensual (gold)
seed codigos_wmo ─► stg_codigos_clima ─────────────────┘   feat_clima_diario (gold · features)
```

In [ ]:
dbt('ls', '--resource-type', 'model', '--resource-type', 'seed', '--resource-type', 'test')

## 2. Build completo: seeds + modelos + tests en orden de dependencias

In [ ]:
dbt('build')

## 3. La gold

In [ ]:
import duckdb
con = duckdb.connect(str(RAIZ / "dbt" / "warehouse.duckdb"), read_only=True)
print("silver:", con.sql("select count(*) from stg_clima_diario").fetchone()[0], "filas (días)")
print("gold  :", con.sql("select count(*) from fct_clima_mensual").fetchone()[0], "filas (meses)")
display(con.sql("select mes, temperatura_media_c, anomalia_temperatura_c, lluvia_total_mm, lluvia_referencia_mm, dias_con_lluvia, fuente_del_dato from fct_clima_mensual order by mes desc limit 12").df())
con.close()

## 4. Demo: romper un dato y ver el test fallar
Copiamos bronze a una tabla **temporal** (no tocamos la real), le metemos un día con la mínima mayor que la máxima
y corremos dbt contra esa copia. `assert_temperaturas_coherentes` tiene que fallar.

In [ ]:
import tempfile, pandas as pd
from deltalake import DeltaTable, write_deltalake
tmp = tempfile.mkdtemp()
malo = DeltaTable(entorno.BRONZE, storage_options=entorno.STORAGE_OPTIONS).to_pandas()
i = malo.index[malo.fecha == "2026-03-15"][0]
malo.loc[i, ["temperature_2m_min", "temperature_2m_max"]] = [35.0, 12.0]     # dato malo
write_deltalake(f"{tmp}/bronze", malo)
codigo = dbt("build", "--select", "stg_clima_diario+", env={"BRONZE_PATH": f"{tmp}/bronze", "DUCKDB_PATH": f"{tmp}/prueba.duckdb"})
print("código de salida:", codigo, "(distinto de 0 = el test detectó el problema)")

## 5. Documentación y linaje
`dbt docs generate` produce el catálogo y el grafo de linaje (`dbt/target/index.html`).

In [ ]:
dbt("docs", "generate")
print("Abrir:", RAIZ / "dbt" / "target" / "index.html")